In [26]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error,mean_absolute_error,r2_score

In [27]:
df= pd.read_csv("calories.csv")

In [28]:
df.describe()

,User_ID,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
count,1.500000e+04,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,1.497736e+07,42.789800,174.465133,74.966867,15.530600,95.518533,40.025453,89.539533
std,2.872851e+06,16.980264,14.258114,15.035657,8.319203,9.583328,0.779230,62.456978
min,1.000116e+07,20.000000,123.000000,36.000000,1.000000,67.000000,37.100000,1.000000
25%,1.247419e+07,28.000000,164.000000,63.000000,8.000000,88.000000,39.600000,35.000000
50%,1.499728e+07,39.000000,175.000000,74.000000,16.000000,96.000000,40.200000,79.000000
75%,1.744928e+07,56.000000,185.000000,87.000000,23.000000,103.000000,40.600000,138.000000
max,1.999965e+07,79.000000,222.000000,132.000000,30.000000,128.000000,41.500000,314.000000


In [29]:
df = df.drop(["User_ID"],axis=1)

In [30]:
df.isnull().sum()

Gender        0
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
Calories      0
dtype: int64

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Gender      15000 non-null  str    
 1   Age         15000 non-null  int64  
 2   Height      15000 non-null  float64
 3   Weight      15000 non-null  float64
 4   Duration    15000 non-null  float64
 5   Heart_Rate  15000 non-null  float64
 6   Body_Temp   15000 non-null  float64
 7   Calories    15000 non-null  float64
dtypes: float64(6), int64(1), str(1)
memory usage: 937.6 KB


In [32]:
df.head()

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [33]:
df[df["Calories"] < 100].count()

Gender        8892
Age           8892
Height        8892
Weight        8892
Duration      8892
Heart_Rate    8892
Body_Temp     8892
Calories      8892
dtype: int64

In [34]:
df[(df["Calories"] > 100) & (df["Calories"] < 200)].count()

Gender        5265
Age           5265
Height        5265
Weight        5265
Duration      5265
Heart_Rate    5265
Body_Temp     5265
Calories      5265
dtype: int64

In [35]:
df[df["Calories"] > 200].count()

Gender        744
Age           744
Height        744
Weight        744
Duration      744
Heart_Rate    744
Body_Temp     744
Calories      744
dtype: int64

In [36]:
def activity_from_calories(row):
    if row["Calories"] < 100:
        return "Light"
    elif row["Calories"] < 200:
        return "Moderate"
    else:
        return "Intense"


In [37]:
df["Activity_Type"] = df.apply(activity_from_calories, axis=1)

In [38]:
df

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Activity_Type
0,male,68,190.0,94.0,29.0,105.0,40.8,231.0,Intense
1,female,20,166.0,60.0,14.0,94.0,40.3,66.0,Light
2,male,69,179.0,79.0,5.0,88.0,38.7,26.0,Light
3,female,34,179.0,71.0,13.0,100.0,40.5,71.0,Light
4,female,27,154.0,58.0,10.0,81.0,39.8,35.0,Light
...,...,...,...,...,...,...,...,...,...
14995,female,20,193.0,86.0,11.0,92.0,40.4,45.0,Light
14996,female,27,165.0,65.0,6.0,85.0,39.2,23.0,Light
14997,female,43,159.0,58.0,16.0,90.0,40.1,75.0,Light
14998,male,78,193.0,97.0,2.0,84.0,38.3,11.0,Light


In [39]:
df[df["Activity_Type"]=="Intense"].count()

Gender           782
Age              782
Height           782
Weight           782
Duration         782
Heart_Rate       782
Body_Temp        782
Calories         782
Activity_Type    782
dtype: int64

In [40]:
df[df["Activity_Type"]=="Light"].count()

Gender           8892
Age              8892
Height           8892
Weight           8892
Duration         8892
Heart_Rate       8892
Body_Temp        8892
Calories         8892
Activity_Type    8892
dtype: int64

In [41]:
df[df["Activity_Type"]=="Moderate"].count()


Gender           5326
Age              5326
Height           5326
Weight           5326
Duration         5326
Heart_Rate       5326
Body_Temp        5326
Calories         5326
Activity_Type    5326
dtype: int64

In [42]:
Q1 = df["Calories"].quantile(0.25)
Q3 = df["Calories"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["Calories"] < lower_bound) | (df["Calories"] > upper_bound)]

print("Number of outliers:", outliers.shape[0])

Number of outliers: 4


In [43]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(numeric_cols)

['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Calories']


In [44]:
outlier_summary = {}

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    outlier_summary[col] = outliers.shape[0]

print("Outliers in each column:")
print(outlier_summary)

Outliers in each column:
{'Age': 0, 'Height': 14, 'Weight': 6, 'Duration': 0, 'Heart_Rate': 1, 'Body_Temp': 369, 'Calories': 4}


In [46]:
if "Body_Temp" in numeric_cols:
    numeric_cols.remove("Body_Temp")


Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

condition = ~((df[numeric_cols] < (Q1 - 1.5 * IQR)) | 
              (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)

df = df[condition]

print("Original Shape:", df.shape)
#print("Cleaned Shape:", df_clean.shape)

Original Shape: (14979, 9)


In [47]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    outlier_summary[col] = outliers.shape[0]

print("Outliers in each column:")
print(outlier_summary)

Outliers in each column:
{'Age': 0, 'Height': 0, 'Weight': 0, 'Duration': 0, 'Heart_Rate': 0, 'Body_Temp': 369, 'Calories': 0}


In [48]:
df.head()

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Activity_Type
0,male,68,190.0,94.0,29.0,105.0,40.8,231.0,Intense
1,female,20,166.0,60.0,14.0,94.0,40.3,66.0,Light
2,male,69,179.0,79.0,5.0,88.0,38.7,26.0,Light
3,female,34,179.0,71.0,13.0,100.0,40.5,71.0,Light
4,female,27,154.0,58.0,10.0,81.0,39.8,35.0,Light


In [49]:
nominal_cols = df.select_dtypes(include="object").columns.tolist()

C:\Users\ronak\AppData\Local\Temp\ipykernel_21060\3346200123.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  nominal_cols = df.select_dtypes(include="object").columns.tolist()


In [50]:
nominal_cols

['Gender', 'Activity_Type']

In [51]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop="first", sparse_output=False)

encoded_data = encoder.fit_transform(df[["Gender", "Activity_Type"]])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(["Gender", "Activity_Type"]),
    index=df.index  
)



In [52]:
df = pd.concat([df, encoded_df], axis=1)

In [53]:
df.drop(["Gender", "Activity_Type"], axis=1, inplace=True)

In [54]:
df

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Gender_male,Activity_Type_Light,Activity_Type_Moderate
0,68,190.0,94.0,29.0,105.0,40.8,231.0,1.0,0.0,0.0
1,20,166.0,60.0,14.0,94.0,40.3,66.0,0.0,1.0,0.0
2,69,179.0,79.0,5.0,88.0,38.7,26.0,1.0,1.0,0.0
3,34,179.0,71.0,13.0,100.0,40.5,71.0,0.0,1.0,0.0
4,27,154.0,58.0,10.0,81.0,39.8,35.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
14995,20,193.0,86.0,11.0,92.0,40.4,45.0,0.0,1.0,0.0
14996,27,165.0,65.0,6.0,85.0,39.2,23.0,0.0,1.0,0.0
14997,43,159.0,58.0,16.0,90.0,40.1,75.0,0.0,1.0,0.0
14998,78,193.0,97.0,2.0,84.0,38.3,11.0,1.0,1.0,0.0


In [55]:
X = df.drop(["Calories"],axis=1)

In [56]:
x_column = X.columns.tolist()

In [57]:
Y = df["Calories"]

In [58]:
scaler = StandardScaler()
X= scaler.fit_transform(X)

In [59]:
X = pd.DataFrame(X , columns=x_column)

In [60]:
X

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Gender_male,Activity_Type_Light,Activity_Type_Moderate
0,1.485295,1.094886,1.271874,1.619928,0.991249,0.994181,1.007438,-1.206805,-0.742363
1,-1.341949,-0.596208,-0.998591,-0.183814,-0.157946,0.352463,-0.992617,0.828634,-0.742363
2,1.544196,0.319802,0.270199,-1.266060,-0.784780,-1.701035,1.007438,0.828634,-0.742363
3,-0.517336,0.319802,-0.264028,-0.304064,0.468887,0.609150,-0.992617,0.828634,-0.742363
4,-0.929643,-1.441755,-1.132147,-0.664812,-1.516087,-0.289255,-0.992617,0.828634,-0.742363
...,...,...,...,...,...,...,...,...,...
14974,-1.341949,1.306273,0.737647,-0.544563,-0.366891,0.480807,-0.992617,0.828634,-0.742363
14975,-0.929643,-0.666670,-0.664699,-1.145810,-1.098197,-1.059317,-0.992617,0.828634,-0.742363
14976,0.012772,-1.089444,-1.132147,0.056685,-0.575836,0.095776,-0.992617,0.828634,-0.742363
14977,2.074304,1.306273,1.472209,-1.626808,-1.202670,-2.214410,1.007438,0.828634,-0.742363


In [61]:
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=23)

In [62]:
lr_model = LinearRegression()
rf_model = RandomForestRegressor(random_state=42)
xgb_model = XGBRegressor(random_state=42)

lr_model.fit(x_train, y_train)
rf_model.fit(x_train, y_train)
xgb_model.fit(x_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [63]:
models = {
    "Linear Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    print(name)
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("R2:", r2_score(y_test, y_pred))
    print("--------------------")

Linear Regression
MAE: 6.5702211357791125
R2: 0.9808854218640397
--------------------
Random Forest
MAE: 1.8559679572763683
R2: 0.9978825088153349
--------------------
XGBoost
MAE: 1.4939377697032985
R2: 0.9988231644297169
--------------------


In [64]:
print("check overfitting :")
print("Train R2:", xgb_model.score(x_train, y_train))
print("Test R2:", xgb_model.score(x_test, y_test))

check overfitting :
Train R2: 0.9995965082625755
Test R2: 0.9988231644297169


In [65]:
feature_columns = X.columns

In [ ]:
import pandas as pd
import numpy as np

def predict_calories(age, gender, height, weight, duration, heart_rate, body_temp, activity_type):
    
    
    input_data = pd.DataFrame({
        "Age": [age],
        "Gender": [gender],
        "Height": [height],
        "Weight": [weight],
        "Duration": [duration],
        "Heart_Rate": [heart_rate],
        "Body_Temp": [body_temp],
        "Activity_Type": [activity_type]
    })
    
    encoded = encoder.transform(input_data[["Gender", "Activity_Type"]])
    
    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(["Gender", "Activity_Type"])
    )

    input_data = input_data.drop(["Gender", "Activity_Type"], axis=1)
    
    
    input_data = pd.concat([input_data.reset_index(drop=True),
                            encoded_df.reset_index(drop=True)], axis=1)
    
        
    input_data = input_data[feature_columns]
    
    
    input_scaled = scaler.transform(input_data)
    
    
    prediction = model.predict(input_scaled)
    
    return round(prediction[0], 2)

In [74]:
result = predict_calories(
    age=69,
    gender="male",
    height=179.0,
    weight=79.0,
    duration=5.0,
    heart_rate=88.0,
    body_temp=38.7,
    activity_type="Light"
)

print("Predicted Calories Burned:", result)

Predicted Calories Burned: 26.1


In [68]:
data = pd.read_csv("calories.csv")
data.head()

,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,16180408,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,17771927,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [69]:
import pickle
pickle.dump(model, open("model.pkl", "wb"))

In [70]:
pickle.dump(scaler, open("scaler.pkl", "wb"))

In [71]:

pickle.dump(encoder, open("encoder.pkl", "wb"))

In [72]:
pickle.dump(feature_columns, open("features.pkl", "wb"))

In [73]:
print(encoder.get_feature_names_out(["Gender", "Activity_Type"]))

['Gender_male' 'Activity_Type_Light' 'Activity_Type_Moderate']
